In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_Fact_ObjectStates V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/object_permission" # ← Change source path
TARGET_PATH = "abfss://Gold/Fact_ObjectStates" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 3, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_Fact_ObjectStates V2...
🚀 Starting ntk_Sil2Gld_Fact_ObjectStates V2


In [2]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")  
# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "object_permission.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Fact_ObjectStates.parquet"
print(f"Variables created and session started.")



StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 4, Finished, Available, Finished)

Variables created and session started.


In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")

StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 5, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/05/object_permission.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Fact_ObjectStates.parquet


In [9]:
# mssparkutils.notebook.run("nbk_dimuser_validations", 60)

StatementMeta(, 4844c273-3598-40a7-b524-026854cd5cc6, 11, Finished, Available, Finished)

In [4]:

# ===== READ PROCESS =====
try:
    # Read the Parquet file from Silver layer
    print("Reading from Silver layer...")
    df_silver = spark.read.parquet(full_source_path)
    print(f"Input file read with total records: {df_silver.count()}")
    
    df_silver.show(1)
    df_silver.printSchema()
    # STAGE 1: PRE-VALIDATION GATE - CRITICAL CHECKPOINT
    # validation_passed, validation_report = step1_validate_source_quality(df_silver)

    # STOP PIPELINE IF CRITICAL VALIDATION FAILURES
    # if not validation_passed:
    #     print("\n🛑 ETL PIPELINE STOPPED - Critical validation failures detected")
    #     print("Fix source data issues before proceeding")
    #     raise Exception("Pre-validation failed - ETL pipeline terminated")
        
    # print("\n🚀 Pre-validation passed - Proceeding with transformation...")

except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("File Read Successful!")


StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 6, Finished, Available, Finished)

Reading from Silver layer...
Input file read with total records: 42
+--------------------+---------------+----------+-------------+--------------------+--------------------+------------+--------------------+--------------------+---------------+--------------------+----------------+---------------------+----------+---------------+--------------+-------------------+----------------+-----------------+-------------+--------------------+--------------------+--------------------+-----------+--------------------------+-------+-------+---------+----------+----------------+--------------------+---------------+----------------+---------------+---------------------+-------------------+----------------------+--------------------+-------------+--------------------+--------------------+---------------+----------+-------+-----------+-----------+---------------+------------+----------+-----------+----------------+---------+--------------+--------------------+-------------+--------------------+--------

In [5]:
 # Creating UsereKey hashvalue
from pyspark.sql.functions import col, when, sha2, concat

# Creating hash key from combined columns
df_silver = df_silver.withColumn(
    "AccessKey",
    when(
        col("ObjectKey").isNotNull() | col("UserKey").isNotNull() | col("GroupKey").isNotNull() | col("PermissionKey").isNotNull(),
        sha2(concat(
            col("ObjectKey"), 
            col("UserKey"), 
            col("GroupKey"), 
            col("PermissionKey")
        ), 256)
    ).otherwise(None)
)
  
print(f"Hash key created")

StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 7, Finished, Available, Finished)

Hash key created


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ===== ETL TRANSFORMATION PLAN =====

def transform_silver_to_gold(df_silver):
    """
    Transform Silver layer Dim_User to Gold layer reporting format
    """
    
    # 1. COLUMN SELECTION & DIRECT MAPPING
    df_transformed = df_silver.select(
        col("AccessKey").alias("AccessID"),
        col("ObjectKey").alias("ObjectID"),
        col("UserKey").alias("UserID"),
        col("GroupKey").alias("GroupID"),
        col("PermissionKey").alias("PermissionID"),
        col("InheritedFrom").alias("GrantedVia"),
        col("IsExternal").alias("IsSharedExternally"),
        col("SnapshotDate")             
    )
    
    # 2. ADD MISSING COLUMNS WITH DEFAULT VALUES
    df_gold = df_transformed \
        .withColumn("ChangedType", lit(None).cast(StringType())) \
        .withColumn("EffectivePermission", lit(None).cast(StringType())) \
        .withColumn("IsCurrent", lit(None).cast(StringType())) \
        .withColumn("IsNewRecord", lit(None).cast(StringType())) \
        .withColumn("IsDeletedRecord", lit(None).cast(StringType())) \
        .withColumn("Owner_UserKey", lit(None).cast(StringType())) \
        .withColumn("LastModifiedBy", lit(None).cast(StringType())) \
        .withColumn("LastModifiedDate", lit(None).cast(StringType())) \
        .withColumn("LastAccessedDate", lit(None).cast(StringType())) \
        .withColumn("VersionHis", lit(None).cast(StringType())) \
        .withColumn("SensitivityLabel", lit(None).cast(StringType())) \
        .withColumn("Classification", lit(None).cast(StringType())) \
        .withColumn("DCLocation", lit(None).cast(StringType())) \
        .withColumn("RetentionPolicyKey", lit(None).cast(StringType())) \
        .withColumn("RetentionPolicy", lit(None).cast(StringType())) \
        .withColumn("ITARControlled", lit(None).cast(StringType())) \
        .withColumn("RequiresUSPersonOnly", lit(None).cast(StringType())) \
        .withColumn("LastAccessedDate", lit(None).cast(StringType())) \
        .withColumn("ShareLinkURL", lit(None).cast(StringType())) \
        .withColumn("ShareLinkType", lit(None).cast(StringType())) \
        .withColumn("ShareLinkExpirationDate", lit(None).cast(StringType())) \
        .withColumn("ShareLinkCreatedBy", lit(None).cast(StringType())) \
        .withColumn("ShareLinkCreatedDate", lit(None).cast(StringType())) \
        .withColumn("DataOrigin", lit(None).cast(StringType())) \
        .withColumn("IsOrphaned", lit(None).cast(StringType())) \
        .withColumn("IsUnlabeled", lit(None).cast(StringType())) \
        .withColumn("IsUnprotected", lit(None).cast(StringType())) \
        .withColumn("DataSource", lit(None).cast(StringType()))   

    # 3. REORDER COLUMNS TO MATCH OUTPUT STRUCTURE
    df_final = df_gold.select(
        "AccessID",
        "ObjectID",
        "UserID",
        "GroupID",
        "PermissionID",
        "IsDeletedRecord",
        "GrantedVia",
        "Owner_UserKey",
        "VersionHis",
        "SensitivityLabel",
        "Classification",
        "DCLocation",
        "RetentionPolicyKey",
        "RetentionPolicy",
        "ITARControlled",
        "RequiresUSPersonOnly",
        "IsSharedExternally",
        "ShareLinkURL",
        "ShareLinkType",
        "ShareLinkExpirationDate",
        "ShareLinkCreatedBy",
        "ShareLinkCreatedDate",
        "DataOrigin",
        "IsOrphaned",
        "IsUnlabeled",
        "IsUnprotected",
        "DataSource",
        "LastModifiedBy",
        "LastModifiedDate",
        "LastAccessedDate",
        "SnapshotDate"
    )
    
    return df_final

# Apply transformation
df_gold_ready = transform_silver_to_gold(df_silver)

# Show results
print("Gold layer transformation completed!")
df_gold_ready.show(5, truncate=False)
df_gold_ready.printSchema()

StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 8, Finished, Available, Finished)

Gold layer transformation completed!
+--------+----------------------------------------------------------------+------+----------------------------------------------------------------+----------------------------------------------------------------+---------------+----------+-------------+----------+----------------+--------------+----------+------------------+---------------+--------------+--------------------+------------------+------------+-------------+-----------------------+------------------+--------------------+----------+----------+-----------+-------------+----------+--------------+----------------+----------------+--------------------------+
|AccessID|ObjectID                                                        |UserID|GroupID                                                         |PermissionID                                                    |IsDeletedRecord|GrantedVia|Owner_UserKey|VersionHis|SensitivityLabel|Classification|DCLocation|RetentionPolicyKey|RetentionPoli

In [7]:
# ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")
    
    df_gold_ready.write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(full_target_path)
    
    print(f"Successfully written to: {full_target_path}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 9, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Fact_ObjectStates.parquet
Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Fact_ObjectStates.parquet
Verification - Target record count: 42
Process completed!


In [ ]:
df_verify.printSchema()

StatementMeta(, 4844c273-3598-40a7-b524-026854cd5cc6, -1, Cancelled, , Cancelled)

In [8]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = df_silver.count()
    rows_written = df_gold_ready.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, bed94653-ef51-4ab1-af68-66f29ecd3ec0, 10, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_Fact_ObjectStates V2...
✅ ntk_Sil2Gld_Fact_ObjectStates V2 completed successfully (139s)
🎉 ntk_Sil2Gld_Fact_ObjectStates V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 42 → 42
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_Fact_ObjectStates V2:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_Fact_ObjectStates V2 logging completed!
